# Dashboard GAC - Analisis Univariado (Despliegue)

**Instrucciones:**
1. Ejecuta la celda 1 para instalar las librerias.
2. Ejecuta la celda 2 para crear el archivo `app.py`.
3. Ejecuta la celda 3 para correr el dashboard localmente.

Asegurate de que los archivos `Bitacora_Piso_Limpia.csv` y `Leads_Reales_Limpia.csv`
esten en la misma carpeta que este notebook.

In [ ]:
%pip install streamlit plotly

In [ ]:
%%writefile app.py
# Dashboard GAC - Analisis Univariado
# Diseño plano y sencillo (colores solidos basicos)

import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# =========================================================
# CONFIGURACION DE PAGINA
# =========================================================
st.set_page_config(
    page_title="Analisis GAC",
    page_icon="📊",
    layout="wide"
)

# =========================================================
# CARGA DE DATOS
# =========================================================
@st.cache_data
def load_data():
    df_bitacora = pd.read_csv("Bitacora_Piso_Limpia.csv")
    df_leads = pd.read_csv("Leads_Reales_Limpia.csv")
    return df_bitacora, df_leads

df_bitacora, df_leads = load_data()

# =========================================================
# PALETA DE COLORES PLANA (basica, sin gradientes)
# =========================================================
COLORES = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
           "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"]

# =========================================================
# BARRA LATERAL
# =========================================================
st.sidebar.title("ANALISIS GAC")
st.sidebar.markdown("---")

vista = st.sidebar.selectbox(
    "Selecciona una vista:",
    ["Extraccion de Caracteristicas",
     "Solicitudes de Credito por Asesor",
     "Visitas por Mes"]
)

st.sidebar.markdown("---")
st.sidebar.write("Base Bitacora de Piso: %d registros" % len(df_bitacora))
st.sidebar.write("Base Leads Reales: %d registros" % len(df_leads))

# =========================================================
# VISTA 1: EXTRACCION DE CARACTERISTICAS
# (Pareto de Asesor, Dona de Prueba de Manejo,
#  Barras de Efectividad de Leads)
# =========================================================
if vista == "Extraccion de Caracteristicas":

    st.title("Extraccion de Caracteristicas - Analisis Univariado")

    col1, col2 = st.columns(2)

    # ---------- GRAFICA 1: PARETO DE ASESOR ----------
    with col1:
        st.subheader("Visitas por Asesor (Pareto)")

        tabla_asesor = df_bitacora["Asesor"].value_counts().reset_index()
        tabla_asesor.columns = ["categorias", "frecuencia"]
        tabla_asesor["porcentaje"] = (
            tabla_asesor["frecuencia"] / tabla_asesor["frecuencia"].sum() * 100
        ).round(1)
        tabla_asesor["porcentaje_acumulado"] = tabla_asesor["porcentaje"].cumsum().round(1)

        fig1 = go.Figure()
        fig1.add_trace(go.Bar(
            x=tabla_asesor["categorias"],
            y=tabla_asesor["frecuencia"],
            name="Frecuencia",
            marker_color="#1f77b4"
        ))
        fig1.add_trace(go.Scatter(
            x=tabla_asesor["categorias"],
            y=tabla_asesor["porcentaje_acumulado"],
            name="% acumulado",
            yaxis="y2",
            mode="lines+markers",
            line=dict(color="#d62728", width=2)
        ))
        fig1.update_layout(
            xaxis_title="Asesor",
            yaxis=dict(title="Visitas"),
            yaxis2=dict(overlaying="y", side="right",
                        range=[0, 105], title="% acumulado"),
            height=400,
            plot_bgcolor="white",
            paper_bgcolor="white",
            showlegend=True
        )
        st.plotly_chart(fig1, use_container_width=True)

    # ---------- GRAFICA 2: DONA DE PRUEBA DE MANEJO ----------
    with col2:
        st.subheader("Visitas con y sin Prueba de Manejo")

        tabla_pdm = df_bitacora["Prueba_Manejo"].value_counts().reset_index()
        tabla_pdm.columns = ["categorias", "frecuencia"]

        fig2 = px.pie(
            data_frame=tabla_pdm,
            names="categorias",
            values="frecuencia",
            hole=0.4,
            color="categorias",
            color_discrete_map={"Sí": "#1f77b4", "No": "#ff7f0e"}
        )
        fig2.update_traces(textinfo="percent+label")
        fig2.update_layout(height=400, paper_bgcolor="white")
        st.plotly_chart(fig2, use_container_width=True)

    # ---------- GRAFICA 3: EFECTIVIDAD DE LEADS POR MES ----------
    st.subheader("Efectividad de Leads por Mes")

    fig3 = px.bar(
        data_frame=df_leads,
        x="Periodo",
        y="Efectividad_Leads",
        color="Nivel_Efectividad",
        color_discrete_map={"Alta": "#2ca02c", "Media": "#1f77b4", "Baja": "#d62728"},
        category_orders={"Nivel_Efectividad": ["Baja", "Media", "Alta"]}
    )
    fig3.add_hline(y=55, line_dash="dash", line_color="gray")
    fig3.add_hline(y=70, line_dash="dash", line_color="gray")
    fig3.update_layout(
        xaxis_title="Periodo",
        yaxis_title="Efectividad (%)",
        height=450,
        plot_bgcolor="white",
        paper_bgcolor="white"
    )
    st.plotly_chart(fig3, use_container_width=True)

    # ---------- TABLA RESUMEN ----------
    st.subheader("Tabla de frecuencias - Asesor")
    st.dataframe(tabla_asesor, use_container_width=True)

# =========================================================
# VISTA 2: SOLICITUDES DE CREDITO POR ASESOR (SDC)
# =========================================================
elif vista == "Solicitudes de Credito por Asesor":

    st.title("Solicitudes de Credito (SDC) por Asesor")

    # Top 5 asesores con mas registros
    top_asesores = (
        df_bitacora.loc[df_bitacora["Asesor"] != "Otros", "Asesor"]
        .value_counts()
        .head(5)
        .index
    )

    sdc_asesor = df_bitacora[df_bitacora["Asesor"].isin(top_asesores)].copy()
    sdc_asesor["SDC"] = sdc_asesor["SDC"].map({1: "Si solicito", 0: "No solicito"})

    sdc_por_asesor = (
        sdc_asesor.groupby(["Asesor", "SDC"])
        .size()
        .reset_index(name="Clientes")
    )

    fig_sdc = px.bar(
        sdc_por_asesor,
        x="Asesor",
        y="Clientes",
        color="SDC",
        barmode="group",
        text="Clientes",
        color_discrete_map={"Si solicito": "#1f77b4", "No solicito": "#ff7f0e"}
    )
    fig_sdc.update_traces(textposition="outside")
    fig_sdc.update_layout(
        xaxis_title="Asesor",
        yaxis_title="Numero de clientes",
        legend_title="SDC",
        height=450,
        plot_bgcolor="white",
        paper_bgcolor="white"
    )
    st.plotly_chart(fig_sdc, use_container_width=True)

    st.subheader("Detalle")
    st.dataframe(sdc_por_asesor, use_container_width=True)

# =========================================================
# VISTA 3: VISITAS POR MES
# =========================================================
elif vista == "Visitas por Mes":

    st.title("Visitas Registradas por Mes")

    visitas_mes = (
        df_bitacora.groupby(["Mes_Num", "Mes"], as_index=False)
        .size()
        .rename(columns={"size": "Visitas"})
        .sort_values("Mes_Num")
    )

    fig_mes = px.bar(
        visitas_mes,
        x="Mes",
        y="Visitas",
        text="Visitas",
        color_discrete_sequence=["#1f77b4"]
    )
    fig_mes.update_traces(textposition="outside")
    fig_mes.update_layout(
        xaxis_title="Mes",
        yaxis_title="Numero de visitas",
        showlegend=False,
        height=450,
        plot_bgcolor="white",
        paper_bgcolor="white"
    )
    st.plotly_chart(fig_mes, use_container_width=True)

    st.subheader("Detalle")
    st.dataframe(visitas_mes[["Mes", "Visitas"]], use_container_width=True)


In [ ]:
! streamlit run app.py